# Proyecto de Clasificación y Reducción de Dimensionalidad

## 1. Configuración del Entorno

In [ ]:
# -*- coding: utf-8 -*-
import os, sys, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib

ruta_modulo = os.path.abspath(os.path.join('..'))
if ruta_modulo not in sys.path: sys.path.append(ruta_modulo)

from src.data import loader as cargador
from src.reduction import reducers as reductores
from src.models import training as entrenamiento
from src.evaluation import metrics as evaluacion
from src.optimization import selection as seleccion
from src.utils import helpers as ayudantes

SEMILLA = 42
np.random.seed(SEMILLA)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook')
NOMBRES_CLASES = ['Camiseta', 'Pantalón', 'Suéter', 'Vestido', 'Abrigo', 'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín']
print("Entorno configurado.")

## 2. Carga de Datos y Artefactos

In [ ]:
rutas_crudos, _, _ = cargador.ejecutar_pipeline_carga(sobrescribir=False)
y_train_crudo = np.load(rutas_crudos['y_train'])
y_test_crudo = np.load(rutas_crudos['y_test'])
x_crudo_completo = np.vstack((np.load(rutas_crudos['X_train']), np.load(rutas_crudos['X_test'])))
y_crudo_completo = np.concatenate((y_train_crudo, y_test_crudo))
print("Datos y etiquetas cargados.")

## Bloque C: Clasificación, Evaluación y Selección

### 11. Análisis de Resultados y Selección de Candidatos

In [ ]:
df_resultados_completos = pd.read_csv('../outputs/tables/summary_results.csv')
objetivos_pareto = {'f1_macro': 'max', 'n_componentes': 'min'}
frente_pareto = seleccion.calcular_frente_pareto(df_resultados_completos, objetivos_pareto)
f1_max = df_resultados_completos['f1_macro'].max()
candidatos_finales = frente_pareto[frente_pareto['f1_macro'] >= 0.98 * f1_max]

if not candidatos_finales.empty:
    candidatos_finales = candidatos_finales.sort_values(['n_componentes', 'tiempo_entrenamiento_s'], ascending=[True, True])
    modelo_final_seleccionado = candidatos_finales.iloc[0]
else:
    modelo_final_seleccionado = frente_pareto.iloc[0]

print("=== Modelo Final Seleccionado ===")
display(modelo_final_seleccionado.to_frame('Valor'))

### 14. Validación Final del Modelo Seleccionado
Re-entrenamos la pipeline completa (reductor + clasificador) con todos los datos de entrenamiento disponibles y la evaluamos en el conjunto de prueba.

In [ ]:
# Configuración del modelo final
norm_final = modelo_final_seleccionado['normalizador']
metodo_red_final = modelo_final_seleccionado['metodo_reduccion']
n_comp_final = int(modelo_final_seleccionado['n_componentes'])
clf_final_nombre = modelo_final_seleccionado['clasificador']

# Cargar datos normalizados completos
x_train_norm_final, x_test_norm_final, _, _, _ = cargador.cargar_conjunto_procesado(dir_procesados='../data/processed/', nombre_normalizador=norm_final)

# Re-entrenar reductor
reductor_final = reductores.obtener_reductor(metodo_red_final, n_comp_final, semilla_aleatoria=SEMILLA)
x_train_red_final = reductor_final.fit_transform(x_train_norm_final, y_train_crudo) if metodo_red_final == 'lda' else reductor_final.fit_transform(x_train_norm_final)

# Re-entrenar clasificador
clf_final = entrenamiento.obtener_clasificador(clf_final_nombre, semilla_aleatoria=SEMILLA)
clf_final_ajustado, _ = entrenamiento.entrenar_modelo(clf_final, x_train_red_final, y_train_crudo)

# Evaluar en el conjunto de prueba
x_test_red_final = reductor_final.transform(x_test_norm_final)
metricas_finales, cm_final, _, _, _ = evaluacion.evaluar_modelo(clf_final_ajustado, x_test_red_final, y_test_crudo, nombres_clases=NOMBRES_CLASES)

print("\n--- Métricas de Validación Finales ---")
df_metricas_finales = pd.DataFrame([metricas_finales])
display(df_metricas_finales)
df_metricas_finales.to_csv('../outputs/tables/final_validation.csv', index=False)

# Guardar modelos finales
joblib.dump(reductor_final, f'../outputs/models/final_reducer.joblib')
joblib.dump(clf_final_ajustado, f'../outputs/models/final_classifier.joblib')

# Visualizar Matriz de Confusión
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Blues', xticklabels=NOMBRES_CLASES, yticklabels=NOMBRES_CLASES, ax=ax)
ax.set_title('Matriz de Confusión del Modelo Final')
ax.set_xlabel('Predicción')
ax.set_ylabel('Valor Verdadero')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
fig.tight_layout()
ayudantes.guardar_figura(fig, '../outputs/figures/C_matriz_confusion_final')

### 15. Demostración de Predicción
Cargamos los modelos finales guardados y los usamos para predecir la clase de una imagen de muestra del conjunto de prueba.

In [ ]:
# Cargar modelos
reductor_cargado = joblib.load('../outputs/models/final_reducer.joblib')
clasificador_cargado = joblib.load('../outputs/models/final_classifier.joblib')

# Seleccionar una muestra aleatoria del conjunto de prueba
indice_muestra = np.random.randint(0, len(x_test_norm_final))
muestra_norm = x_test_norm_final[indice_muestra].reshape(1, -1)
etiqueta_verdadera = y_test_crudo[indice_muestra]

# Pipeline de predicción
muestra_red = reductor_cargado.transform(muestra_norm)
prediccion_numerica = clasificador_cargado.predict(muestra_red)[0]
prediccion_nombre = NOMBRES_CLASES[prediccion_numerica]

print(f"--- Demostración con muestra de prueba #{indice_muestra} ---")
print(f"Etiqueta Verdadera: {NOMBRES_CLASES[etiqueta_verdadera]} (Clase {etiqueta_verdadera})")
print(f"Predicción del Modelo: {prediccion_nombre} (Clase {prediccion_numerica})")

# Visualizar la imagen
fig, ax = plt.subplots(figsize=(4,4))
# Necesitamos la imagen cruda para visualizarla
x_test_crudo = np.load(rutas_crudos['X_test'])
ax.imshow(x_test_crudo[indice_muestra], cmap='gray')
ax.set_title(f"Verdadero: {NOMBRES_CLASES[etiqueta_verdadera]}\nPredicho: {prediccion_nombre}")
ax.axis('off')
plt.show()

**--- Fin del Proyecto ---**